# Common PyTorch tensor operations


In [1]:
#import pytorch
import torch
import torch.nn as nn

In [2]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2], [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]]) 
print(tensor0d)
print(f"Data type {tensor0d.dtype}")

tensor(1)
Data type torch.int64


In [3]:
#convert 64int ->32 bit float
tensor0d_float=tensor0d.to(torch.float32)
print(f"Data type {tensor0d_float.dtype}")

Data type torch.float32


In [4]:
#shape
print(tensor2d.shape)
#reshape the dim
print(tensor2d.reshape(1,4).shape)
#veiw work same as reshape but it fail data is not contiguous
print(tensor2d.view(1,4).shape)
#taking transpose
print(f"Tensor 2d is {tensor2d}")
print(f"Tensor 2d is after transposition{tensor2d.T}")

torch.Size([2, 2])
torch.Size([1, 4])
torch.Size([1, 4])
Tensor 2d is tensor([[1, 2],
        [3, 4]])
Tensor 2d is after transpositiontensor([[1, 3],
        [2, 4]])


In [5]:
#multiply two tensor using .matmul
print(f"Multiplication of Tensor{tensor1d} and {tensor1d.T} is: {tensor1d.matmul(tensor1d.T)}\n")

#we can also multiply using @
print(f"Multiplication of Tensor{tensor1d} and {tensor1d.T} is: {tensor1d@(tensor1d.T)}")


Multiplication of Tensortensor([1, 2, 3]) and tensor([1, 2, 3]) is: 14

Multiplication of Tensortensor([1, 2, 3]) and tensor([1, 2, 3]) is: 14


C:\Users\shivam\AppData\Local\Temp\ipykernel_25816\1634156497.py:2: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4483.)
  print(f"Multiplication of Tensor{tensor1d} and {tensor1d.T} is: {tensor1d.matmul(tensor1d.T)}\n")


# Pytorch autograd

In PyTorch, it will build a computational graph internally by default if one of its terminal nodes has the requires_grad attribute set to True.
This is useful if we want to compute gradients.

if `requires_grad=True` in that case it will calculate grad with respect to it


<div style="text-align:center">
<img src="./IMG/computation_graph_pytorch.png">
</div>


In [6]:
# Computing of gradients via autograd
import torch.nn.functional as f
from torch.autograd import grad

y=torch.tensor([1.0])  #ground truth
x1=torch.tensor([1.1]) #input tensor
w1=torch.tensor([2.2],requires_grad=True) #trainable weigth  here we make (requries_grad=Ture) so pytoch automatic calculate the gradient
b=torch.tensor([0.0],requires_grad=True)  #trainable bias

z=x1*w1+b
a=torch.sigmoid(z)
loss=f.binary_cross_entropy(a,y)

#calculating gradient
grad_L_w1=grad(loss,w1,retain_graph=True) #here retain_graph=true for future uses other wise pytorch delete after calculation
grad_L_b=grad(loss,b,retain_graph=True)
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


PyTorch provides even more high-level tools to automate this process. For instance, we can call
.backward on the loss, and PyTorch will compute the gradients of all the leaf nodes in
the graph, which will be stored via the tensors’ .grad attributes:


In [7]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = f.binary_cross_entropy(a, y)

#using backward() and using .grad we can acess it
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


# Building Neural Network in pytorch


In [8]:
import torch.nn as nn
#simple neural network
class Neural_Network(nn.Module):
  def __init__(self,input_dim,output_dim):
    super().__init__()
    self.layers=nn.Sequential(
      nn.Linear(input_dim,20),#input
      nn.ReLU(),#activation
      nn.Linear(20,10),#hidden layer
      nn.ReLU(),
      nn.Linear(10,output_dim)#output
    )
  def forward(self,x):
    logits=self.layers(x)
    return logits     

In [9]:
torch.manual_seed(123) # seed mean same initialization all time
model=Neural_Network(50,3)
print(model)

#output
torch.manual_seed(123)
X = torch.rand((1, 50))
with torch.no_grad():
  out = model(X)
print(out)

Neural_Network(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=3, bias=True)
  )
)
tensor([[ 0.0522, -0.3840,  0.0038]])


In [10]:
#total number of trainable parameter
num_param=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(num_param)

#number of parameter
for p in model.parameters():
  pass
print(p)  

#print weight of ffn
print(model.layers[0].weight.shape)
# print(model.layers[0].weight)

1263
Parameter containing:
tensor([ 0.0021, -0.1950, -0.0633], requires_grad=True)
torch.Size([20, 50])


# Data Loader in pytorch


In [11]:
# (x_train,y_train) (x_test,y_test)
X_train = torch.tensor([
          [-1.2, 3.1],
          [-0.9, 2.9],
          [-0.5, 2.6],
          [2.3, -1.1],
          [2.7, -1.5]
          ])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
  [-0.8, 2.8],
  [2.6, -1.6],
  ])
y_test = torch.tensor([0, 1])

In [14]:
#define custum dataset that pass to dataloader
from torch.utils.data import Dataset
class ToyDataset(Dataset):
  def __init__(self,X,y):
    super().__init__()
    self.feature=X
    self.labels=y

  def __getitem__(self,index):
      item_x=self.feature[index]
      item_y=self.labels[index]
      return item_x,item_y

  def __len__(self):
      return len(self.labels)
      # return self.labels.shape[0]  

In [15]:
train_ds=ToyDataset(X_train,y_train)
test_ds=ToyDataset(X_test,y_test)
for i,(feature,label) in enumerate(train_ds):
  print(f"dats{i} has feature :{feature} and label is {label}")

dats0 has feature :tensor([-1.2000,  3.1000]) and label is 0
dats1 has feature :tensor([-0.9000,  2.9000]) and label is 0
dats2 has feature :tensor([-0.5000,  2.6000]) and label is 0
dats3 has feature :tensor([ 2.3000, -1.1000]) and label is 1
dats4 has feature :tensor([ 2.7000, -1.5000]) and label is 1


# Instantiating data loaders for sampling


In [20]:
from torch.utils.data import DataLoader
torch.manual_seed(123)

#loading data using dataloader
train_loader=DataLoader(
  dataset=train_ds,
  batch_size=2,
  shuffle=True,
  num_workers=0,
  drop_last=True
)
test_loader=DataLoader(
  dataset=test_ds,
  batch_size=2,
  shuffle=True,
  num_workers=0,
  drop_last=True #if last batch has not exact batch size then it will droped
)
#printing
for i,(feature,label) in enumerate(train_loader):
  print(f"Batch{i} has feature :{feature} and label is {label}\n")

Batch0 has feature :tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) and label is tensor([1, 0])

Batch1 has feature :tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) and label is tensor([0, 0])



# Training of Neural Network


In [22]:
import torch.nn.functional as f

torch.manual_seed(123)
model=Neural_Network(2,2)
optimizer=torch.optim.AdamW(
  model.parameters(),lr=0.05
)
num_epoches=5
for epoch in range(num_epoches):
  model.train() #set model to trianing mode
  for batch_idx,(feature,labels) in enumerate(train_loader):
    logits=model(feature)
    loss=f.cross_entropy(logits,labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #print loss
    print(f"Epoch is: {epoch:03d}/{num_epoches:03d} | Batch is:{batch_idx:03d}/{len(train_loader):03d} | Training_Loss is {loss:3f}")

Epoch is: 000/005 | Batch is:000/002 | Training_Loss is 0.677634
Epoch is: 000/005 | Batch is:001/002 | Training_Loss is 0.146927
Epoch is: 001/005 | Batch is:000/002 | Training_Loss is 0.413387
Epoch is: 001/005 | Batch is:001/002 | Training_Loss is 0.217171
Epoch is: 002/005 | Batch is:000/002 | Training_Loss is 0.105231
Epoch is: 002/005 | Batch is:001/002 | Training_Loss is 0.053452
Epoch is: 003/005 | Batch is:000/002 | Training_Loss is 0.015953
Epoch is: 003/005 | Batch is:001/002 | Training_Loss is 0.001641
Epoch is: 004/005 | Batch is:000/002 | Training_Loss is 0.000001
Epoch is: 004/005 | Batch is:001/002 | Training_Loss is 0.000205


In [ ]:
#define function to compute accuracy
def compute_accuracy(model,dataloader):
  model.eval()
  correct=0.0
  total_example=0
  for batch_idx,(features,labels) in enumerate(dataloader):
    with torch.no_grad():
      logits=model(features)
    probabilty=torch.softmax(logits,dim=1)  
    # print(probabilty)
    prediction=torch.argmax(probabilty,dim=1)
    # prediction=torch.argmax(logits,dim=1)
    cmp=(prediction==labels)
    correct+=torch.sum(cmp)
    total_example+=len(cmp)
  return (correct/total_example).item()

In [59]:
#evaluating the model
torch.set_printoptions(sci_mode=False) # making easy to visulaize
model.eval() # set to eval mode 
print(f"Training accuracy of model is {compute_accuracy(model,train_loader)}")
print(f"Testing accuracy of model is {compute_accuracy(model,test_loader)}")

tensor([[1.0000, 0.0000],
        [1.0000, 0.0000]])
tensor([[1.0000, 0.0000],
        [0.0001, 0.9999]])
Training accuracy of model is 1.0
tensor([[0.0000, 1.0000],
        [1.0000, 0.0000]])
Testing accuracy of model is 1.0


In [65]:
# Save model weights
torch.save(model.state_dict(), "NN_model.pth")

# Load model weights
model.load_state_dict(torch.load("NN_model.pth"))

# Set evaluation mode
model.eval()

Neural_Network(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=2, bias=True)
  )
)

# Optimizing training performance with GPUs


In [67]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [ ]:
# check gpu is available or not
print(torch.cuda.is_available())

#load input to Gpu using .to()
tensor_1=tensor_1.to("cuda")

# or using to(device) we can also do
device=torch.device("cuda") if torch.cuda.is_available() else "cpu"
tensor_2=tensor_2.to(device)

print(tensor_1 + tensor_2)

True
tensor([5., 7., 9.], device='cuda:0')


In [73]:
#traing using GPU
torch.manual_seed(123)
model = Neural_Network(2, 2)
device = torch.device("cuda")
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
  model.train()
  for batch_idx, (features, labels) in enumerate(train_loader):
    features, labels = features.to(device), labels.to(device)
    logits = model(features)
    loss = f.cross_entropy(logits, labels) # Loss function

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### LOGGING
    print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
    f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
    f" | Train/Val Loss: {loss:.2f}")
   

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.68
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.14
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.28
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.14
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.07


In [ ]:
#define function to compute accuracy on GPu
def compute_accuracy(model,dataloader):
  model.eval()
  correct=0.0
  total_example=0
  for batch_idx,(features,labels) in enumerate(dataloader):
    features=features.to(device)
    labels=labels.to(device)
    with torch.no_grad():
      logits=model(features).to(device)
    probabilty=torch.softmax(logits,dim=1)  
    # print(probabilty)
    prediction=torch.argmax(probabilty,dim=1)
    # prediction=torch.argmax(logits,dim=1)
    cmp=(prediction==labels)
    correct+=torch.sum(cmp)
    total_example+=len(cmp)
  return (correct/total_example).item()

In [83]:
#evaluating the model
device=torch.device("cuda")
torch.set_printoptions(sci_mode=False) # making easy to visulaize
device=torch.device("cuda") if torch.cuda.is_available() else "cpu"
model.eval() # set to eval mode 
print(f"Training accuracy of model is {compute_accuracy(model,train_loader)}")
print(f"Testing accuracy of model is {compute_accuracy(model,test_loader)}")

Training accuracy of model is 1.0
Testing accuracy of model is 1.0


In [ ]:
#import torch
import torch.nn as nn
# Define a linear layer with input size 3 and output size 2
linear = nn.Linear(3, 2)
# Example input
x = torch.tensor([[1.0, 2.0, 3.0]])
output = linear(x)
print("Output:", output)

Output: tensor([[ 0.5072, -1.0431]], grad_fn=<AddmmBackward0>)


In [ ]:
relu = nn.ReLU()

# Example input
x = torch.tensor([[-1.0, 2.0], [3.0, -4.0]])
output = relu(x)
print("Output:", output)

Output: tensor([[0., 2.],
        [3., 0.]])


In [ ]:
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
# Example input (batch of 1, 1 channel, 4x4 image)
x = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
                    [5.0, 6.0, 7.0, 8.0],
                    [9.0, 10.0, 11.0, 12.0],
                    [13.0, 14.0, 15.0, 16.0]]]])
output = max_pool(x)
print("Output shape:", output.shape)  # Shape: (1, 1, 2, 2)
print("Output:", output)

Output shape: torch.Size([1, 1, 2, 2])
Output: tensor([[[[ 6.,  8.],
          [14., 16.]]]])


In [ ]:
flatten = nn.Flatten()
# Example input (batch of 1, 3 channels, 32x32 image)
x = torch.randn(1, 3, 32, 32)
output = flatten(x)
print("Output shape:", output.shape)  # Shape: (1, 3072)

Output shape: torch.Size([1, 3072])


In [ ]:
dropout = nn.Dropout(p=0.5)
# Example input
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
output = dropout(x)
print("Output during training:", output)

Output during training: tensor([[2., 4., 0., 0.]])


In [ ]:
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3)

# Example input (batch of 2, sequence length 4)
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
output = embedding(x)
print("Output shape:", output.shape)  # Shape: (2, 3, 3)
output

Output shape: torch.Size([2, 3, 3])


tensor([[[ 0.1559, -0.5776,  0.0598],
         [ 2.0136, -0.6215,  0.6259],
         [-0.8675, -2.1759,  0.5358]],

        [[-1.7770, -1.3277,  1.7146],
         [-0.2447, -0.4534, -1.3577],
         [ 1.4822,  0.3689, -0.8400]]], grad_fn=<EmbeddingBackward0>)

In [ ]:
import torch.nn as nn

# Define the model using nn.Sequential
model = nn.Sequential(
    nn.Linear(10, 20),  # Input layer: 10 features to 20 neurons
    nn.ReLU(),          # Activation function
    nn.Linear(20, 10),  # Hidden layer: 20 neurons to 10 neurons
    nn.ReLU(),          # Activation function
    nn.Linear(10, 1),   # Output layer: 10 neurons to 1 output
    nn.Sigmoid()        # Sigmoid activation for binary classification
)

print(model)

Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=10, bias=True)
  (3): ReLU()
  (4): Linear(in_features=10, out_features=1, bias=True)
  (5): Sigmoid()
)


In [ ]:
!pip install PyMuPDF

In [ ]:
import fitz
def is_password_protected_pdf(pdf_file_path):
    doc = fitz.Document(pdf_file_path)
    if doc.needs_pass:
        return True
    return False
def is_pdf_text_encrypted(pdf_file_path):
    doc = fitz.Document(pdf_file_path)
    if doc.metadata["encryption"] is not None:
        return True
    return False  

In [ ]:
def decrypt_pdf(pdf_file_path, password):
    doc = fitz.Document(pdf_file_path)
    if doc.authenticate(password):
        file_name = "pdf_decrypted.pdf"
        doc.save(file_name)
        print("\Successfully decrypted PDF")
    else:
        print("\t Password incorrect!! Cannot decrypt PDF!!!")

## Onehot Encoding


In [ ]:
def to_onehot(y, num_classes):
    y_onehot = torch.zeros(y.size(0), num_classes)
    y_onehot.scatter_(1, y.view(-1, 1).long(), 1).float()
    return y_onehot

y = torch.tensor([0, 1, 2, 2])

y_enc = to_onehot(y, 3)

print('one-hot encoding:\n', y_enc)

one-hot encoding:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 0., 1.]])


## Softmax


In [ ]:
Z = torch.tensor( [[-0.3,  -0.5, -0.5],
                   [-0.4,  -0.1, -0.5],
                   [-0.3,  -0.94, -0.5],
                   [-0.99, -0.88, -0.5]])

Z.shape

torch.Size([4, 3])

Next, we convert them to "probabilities" via softmax:

$$P(y=j \mid z^{(i)}) = \sigma_{\text{softmax}}(z^{(i)}) = \frac{e^{z^{(i)}}}{\sum_{j=0}^{k} e^{z_{k}^{(i)}}}.$$


In [ ]:
def softmax(z):
  z_max=torch.max(z,dim=1,keepdim=True).values
  exp_z=torch.exp(z-z_max)
  return exp_z/torch.sum(exp_z,dim=1,keepdim=True)
  
softmax(Z)  

tensor([[0.3792, 0.3104, 0.3104],
        [0.3072, 0.4147, 0.2780],
        [0.4263, 0.2248, 0.3490],
        [0.2668, 0.2978, 0.4354]])

$$\mathcal{L}(\mathbf{W}; \mathbf{b}) = \frac{1}{n} \sum_{i=1}^{n} H(T_i, O_i),$$


$$H(T_i, O_i) = -\sum_m T_i \cdot log(O_i).$$


In [ ]:
def softmax(z):
    return (torch.exp(z.t()) / torch.sum(torch.exp(z), dim=1)).t()

smax = softmax(Z)
print('softmax:\n', smax)

softmax:
 tensor([[0.3792, 0.3104, 0.3104],
        [0.3072, 0.4147, 0.2780],
        [0.4263, 0.2248, 0.3490],
        [0.2668, 0.2978, 0.4354]])


In [ ]:
def to_classlabel(z):
    return torch.argmax(z, dim=1)

print('predicted class labels: ', to_classlabel(smax))
print('true class labels: ', to_classlabel(y_enc))

predicted class labels:  tensor([0, 1, 0, 2])
true class labels:  tensor([0, 1, 2, 2])


In [ ]:
def cross_entropy(softmax, y_target):
    return - torch.sum(torch.log(softmax) * (y_target), dim=1)

xent = cross_entropy(smax, y_enc)
print('Cross Entropy:', xent)

Cross Entropy: tensor([0.9698, 0.8801, 1.0527, 0.8314])


## In pytorch


In [ ]:
import torch.nn.functional as F

In [ ]:
F.nll_loss(torch.log(smax), y, reduction='none')

tensor([0.9698, 0.8801, 1.0527, 0.8314])

In [ ]:
F.cross_entropy(Z, y, reduction='none')

tensor([0.9698, 0.8801, 1.0527, 0.8314])

In [ ]:
F.cross_entropy(Z, y)

tensor(0.9335)

In [ ]:
torch.mean(cross_entropy(smax, y_enc))

tensor(0.9335)